<a href="https://colab.research.google.com/github/YOUR_GITHUB_USER/YOUR_REPO/blob/main/track4_sustainability_greenlens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌱 Track 4 · Sustainability
## GreenLens: An AI Analyst for ESG Reports & Greenwashing

**Coaching session · ~60 min**

### The real-world problem
Companies publish long sustainability (ESG) reports full of bold claims: *"carbon neutral!"*, *"footprint cut by 40%!"*, *"100% sustainable packaging!"*. Investors, banks, regulators, journalists and NGOs need to know:
- What are the **actual numbers** (Scope 1, 2 and 3 emissions, renewable energy)?
- Do the **headline claims match the data** further down?
- Where in the report is the evidence?

Analysts do this by hand, one 100-page PDF at a time. Singapore is also phasing in mandatory climate reporting for listed companies, so the volume of reports is growing fast.

> 📘 **Scope 1** = direct emissions (your own fuel). **Scope 2** = purchased electricity. **Scope 3** = everything in your value chain (suppliers, subcontractors, products). Scope 3 is often 70%+ of the total, and it is where greenwashing hides.

### What you will build
| Step | Technique | What it does |
|---|---|---|
| Index | **Tree index (vectorless RAG)** | Turn the report's headings into a table-of-contents tree, with no embeddings |
| Retrieve | **Reasoning-based navigation** | The LLM reads the table of contents and *chooses* which sections to open, like a human analyst ([PageIndex](https://github.com/VectifyAI/PageIndex) idea) |
| Extract | **Structured output + validation** | Emissions table ➜ pandas, with an automatic check that catches a typo in the report |
| Audit | **LLM-as-judge with evidence** | Each headline claim gets a verdict: Supported, Misleading or Unsupported |

> ⚠️ The company and report in this notebook are **fictional**.

## Step 1: Install Dependencies
- `google-genai`: Google's Gen AI SDK for Gemini (already preinstalled in Colab).
- `gradio`: the web UI for the demo.

No vector database this time, which is the point of this track!

The first line records the versions of a few core packages Colab depends on, so the install below doesn't upgrade them and break Colab. If Colab asks you to **restart the session** after installing, click restart and continue from Step 2.

In [ ]:
# Keep Colab's own versions of these packages to avoid dependency conflicts
!pip freeze | grep -iE '^(google-auth|opentelemetry-(api|sdk))==' > /tmp/colab-constraints.txt
!pip install -q -c /tmp/colab-constraints.txt google-genai gradio==6.29.1

## Step 2: Setup Google Gemini API Key
1. Go to [Google AI Studio](https://aistudio.google.com/).
2. Sign in with your Google account and click **Get API key**.
3. In Colab, click the 🔑 **Secrets** icon on the left sidebar.
4. Create a new secret named `GEMINI_API_KEY` and paste your key.
5. Turn on the **Notebook access** toggle for that secret.

We also define `ask_gemini()`, a small wrapper that waits and retries when the free tier says *"429: too many requests"*. Free-tier keys only allow a few requests per minute, so every team will hit this during the hackathon.

In [ ]:
import time
from google import genai
from google.genai import types
from google.colab import userdata

# Securely load the API key from Colab Secrets
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=GOOGLE_API_KEY)
    print("✅ Gemini API Key configured successfully!")
except Exception as e:
    print("❌ Error: Please ensure you added 'GEMINI_API_KEY' to your Colab Secrets and turned on 'Notebook access'.")
    print("   Details:", e)
    raise

# One place to change the model for the whole notebook
MODEL = "gemini-2.5-flash"

def ask_gemini(contents, config=None, retries=3):
    """Call Gemini, waiting and retrying if we hit the free-tier rate limit."""
    for attempt in range(retries):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            if "429" in str(e) and attempt < retries - 1:
                wait = 20 * (attempt + 1)
                print(f"⏳ Rate limited by the free tier, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

## Step 3: The Sustainability Report
A shortened report for a fictional company, **Lumen Logistics**. Read the CEO message, then look at the data sections. Can you spot the problems yourself before the AI does?

In a real project you would convert a PDF to text first (Gemini can read PDFs directly, or use a library like `pymupdf4llm` to get markdown with headings).

In [ ]:
REPORT = """
# Lumen Logistics Sustainability Report 2025

## 1. Message from our CEO
2025 was a landmark year. We have cut our carbon footprint by 40% since 2022, and I am proud to say Lumen is now a carbon-neutral company.
Our fleet is going electric, and 100% of our packaging is now sustainable. We are on track to meet our 2030 climate target.

## 2. About Lumen Logistics
Lumen provides warehousing and last-mile delivery across Southeast Asia, with 14 warehouses, 1,200 delivery vans and 3,800 employees.
Around 60% of deliveries are handled by subcontracted transport partners.

## 3. Climate and Emissions
We report greenhouse gas emissions following the GHG Protocol, in tonnes of CO2 equivalent (tCO2e).

### 3.1 Emissions by Scope
| Year | Scope 1 | Scope 2 | Scope 3 | Total (Scope 1+2+3) |
|------|---------|---------|---------|---------------------|
| 2022 | 18,400 | 9,800 | 61,000 | 89,200 |
| 2023 | 17,900 | 8,100 | 66,500 | 90,500 |
| 2024 | 16,200 | 6,000 | 71,200 | 93,400 |
| 2025 | 15,100 | 4,300 | 78,900 | 98,300 |

Scope 1 comes mainly from diesel delivery vans. Scope 2 is warehouse electricity.
Scope 3 is mostly subcontracted transport and purchased packaging, and grew with our delivery volumes.

### 3.2 Carbon Offsets
In 2025 we purchased 20,000 tCO2e of forestry carbon credits (avoided deforestation project, 2016 vintage) to offset our Scope 1 and Scope 2 emissions.
The credits have not been independently verified for permanence.

## 4. Energy
Renewable electricity share across our warehouses: 2022: 12%, 2023: 25%, 2024: 41%, 2025: 58%.
Most renewable electricity is from purchased renewable energy certificates (RECs); rooftop solar supplies 9% of electricity.
Electric vans in our fleet: 2022: 20, 2025: 140 (out of 1,200 vans).

## 5. Targets
- Reduce combined Scope 1 and 2 emissions by 50% by 2030, compared with 2022.
- Net zero across all scopes by 2050.
- We have not yet set a Scope 3 reduction target.

## 6. Packaging and Waste
All of our packaging is technically recyclable. Recycled content in our packaging was 35% in 2025 (2022: 18%).
Packaging waste sent to landfill: 2,100 tonnes in 2025.

## 7. People and Safety
Lost-time injury frequency rate (LTIFR) improved from 2.1 to 1.6 between 2022 and 2025. Employee volunteering hours: 12,400.

## 8. Reporting Methodology and Assurance
Scope 1 and 2 data received limited assurance from an independent third party. Scope 3 data is estimated using spend-based
emission factors and has not been assured.
"""
print(f"Report length: {len(REPORT.split())} words")

## Step 4: Build a Tree Index (No Vectors!)
Classic RAG chops a document into chunks and embeds them. That breaks tables apart and loses the document's structure.

Instead, we use the structure the author already gave us: **headings**. Each heading becomes a node in a tree, holding its text and its child sections. Then we print the tree as a **table of contents**, which is exactly what a human analyst looks at first.

In [ ]:
import re

def build_tree(markdown):
    nodes, stack = [], []
    for line in markdown.strip().splitlines():
        m = re.match(r"^(#{1,4})\s+(.*)", line)
        if m:
            node = {"id": f"N{len(nodes):02d}", "level": len(m.group(1)), "title": m.group(2).strip(),
                    "text": "", "children": []}
            while stack and stack[-1]["level"] >= node["level"]:
                stack.pop()
            if stack:
                stack[-1]["children"].append(node["id"])
            stack.append(node)
            nodes.append(node)
        elif nodes:
            nodes[-1]["text"] += line + "\n"
    return {n["id"]: n for n in nodes}

def table_of_contents(tree):
    lines = []
    for n in tree.values():
        preview = " ".join(n["text"].split())[:80]
        lines.append(f'{"  " * (n["level"] - 1)}[{n["id"]}] {n["title"]}' + (f" | {preview}..." if preview else ""))
    return "\n".join(lines)

def node_text(tree, node_id, seen=None):
    """Return a section's text plus all of its sub-sections."""
    seen = set() if seen is None else seen
    if node_id in seen or node_id not in tree:
        return ""
    seen.add(node_id)
    n = tree[node_id]
    return f'### {n["title"]} [{node_id}]\n{n["text"]}' + "".join(node_text(tree, c, seen) for c in n["children"])

TREE = build_tree(REPORT)
TOC = table_of_contents(TREE)
print(TOC)

## Step 5: Reasoning-Based Retrieval
Now the LLM **navigates**. It reads only the table of contents (cheap!) and decides which sections to open, explaining why. Then we send only those sections to answer the question.

This works well for questions that don't share words with the answer. *"Is the CEO's 40% claim true?"* needs the emissions **table**, which never mentions "CEO" or "claim". A similarity search can easily miss it; a reasoning analyst won't.

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel, Field

class Navigation(BaseModel):
    reasoning: str = Field(description="One or two sentences: why these sections are likely to hold the answer")
    node_ids: list[str] = Field(description="Up to 3 node IDs to open, e.g. ['N04']")

def navigate(question, tree=None):
    tree = tree or TREE
    prompt = f"""You are an ESG analyst. Here is the table of contents of a sustainability report.
Pick the sections (up to 3) you need to open to answer the question. Remember: headline claims are often in
opening letters, but the real numbers are in data sections, targets and methodology notes.

TABLE OF CONTENTS:
{table_of_contents(tree)}

QUESTION: {question}"""
    nav = ask_gemini(prompt, types.GenerateContentConfig(response_mime_type="application/json", response_schema=Navigation)).parsed
    nav.node_ids = [i for i in nav.node_ids if i in tree]  # never trust IDs blindly
    return nav

def ask_report(question, tree=None):
    tree = tree or TREE
    nav = navigate(question, tree)
    seen = set()
    context = "".join(node_text(tree, i, seen) for i in nav.node_ids)
    prompt = f"""Answer the question using ONLY the report sections below. Cite section IDs like [N04].
If the sections don't contain the answer, say so. Be specific and use numbers.

SECTIONS:
{context}

QUESTION: {question}"""
    return nav, ask_gemini(prompt).text

nav, answer = ask_report("Is the CEO's claim that the carbon footprint fell 40% accurate?")
print("🧭 Sections opened:", nav.node_ids, "\n   because:", nav.reasoning)
print("\n💬", answer)

## Step 6: Extract the Numbers and Validate Them
For charts and calculations we need the emissions table as data. Structured output turns the table into a list of Pydantic objects, and pandas does the maths.

**Validation trick:** the report gives a "Total" column. We ask the model to copy the reported total *as written* and then check `Scope 1 + 2 + 3 == reported total` ourselves. If they disagree, either the extraction or **the report itself** is wrong. Both are worth knowing!

In [ ]:
import pandas as pd

class YearMetrics(BaseModel):
    year: int
    scope1_tco2e: float
    scope2_tco2e: float
    scope3_tco2e: float
    reported_total_tco2e: float = Field(description="The total exactly as printed in the report, even if it looks wrong")
    renewable_electricity_pct: Optional[float] = Field(description="Renewable electricity share in percent, or null")

# Use the tree to find the data sections first, so we don't send the whole report
data_nav = navigate("Annual Scope 1, 2 and 3 emissions by year, and renewable electricity share by year")
data_context = "".join(node_text(TREE, i, set()) for i in data_nav.node_ids)
print("🧭 Data sections:", data_nav.node_ids)

metrics = ask_gemini(
    "Extract one record per year from these report sections. Copy numbers exactly as printed.\n\n" + data_context,
    types.GenerateContentConfig(response_mime_type="application/json", response_schema=list[YearMetrics]),
).parsed

em = pd.DataFrame([m.model_dump() for m in metrics]).set_index("year").sort_index()
em["computed_total"] = em.scope1_tco2e + em.scope2_tco2e + em.scope3_tco2e
em["total_check"] = (em.computed_total == em.reported_total_tco2e).map({True: "✅ ok", False: "❌ mismatch"})
em

In [ ]:
import matplotlib.pyplot as plt

first, last = em.index.min(), em.index.max()
def pct_change(series):
    return round(float((series[last] - series[first]) / series[first] * 100), 1)

s12 = em.scope1_tco2e + em.scope2_tco2e
FACTS = {
    "period": f"{first} to {last}",
    "total_change_pct (all scopes, computed)": pct_change(em.computed_total),
    "scope1_plus_2_change_pct": pct_change(s12),
    "scope3_change_pct": pct_change(em.scope3_tco2e),
    "scope3_share_of_total_latest_pct": round(float(em.scope3_tco2e[last] / em.computed_total[last] * 100), 1),
    "scope1_plus_2_latest_tco2e": float(s12[last]),
    "years_with_total_mismatch": [int(y) for y in em.index[em.total_check != "✅ ok"]],
}
for k, v in FACTS.items():
    print(f"{k:45s} {v}")

fig, ax = plt.subplots(figsize=(8, 4))
em[["scope1_tco2e", "scope2_tco2e", "scope3_tco2e"]].plot(kind="bar", stacked=True, ax=ax, color=["#d95f02", "#7570b3", "#1b9e77"])
ax.set_title("Lumen Logistics emissions by scope (tCO2e)"); ax.set_xlabel(""); plt.xticks(rotation=0)
if em.renewable_electricity_pct.notna().any():
    ax2 = ax.twinx(); ax2.plot(range(len(em)), em.renewable_electricity_pct, "k--o", label="Renewable electricity %")
    ax2.set_ylabel("Renewable %"); ax2.set_ylim(0, 100); ax2.legend(loc="upper left")
plt.tight_layout(); plt.show()

### 🤔 Coaching discussion (5 min)
- Scope 1+2 fell, but the **total rose**. Which number did the CEO quote? Is that lying, or just "choosing the flattering number"?
- Did the validation check catch the **2023 typo** in the report's total column? Would a reader have noticed?
- Scope 3 isn't assured and has no target, yet it's ~80% of emissions. What should an investor ask?

## Step 7: The Greenwashing Auditor
Two LLM calls:
1. **Extract** the headline claims from the CEO message.
2. **Judge** each claim against the evidence: the numbers *we computed with pandas* (not the LLM) plus the relevant report sections.

The judge must quote evidence for every verdict. A verdict without evidence is just an opinion.

In [ ]:
class Claim(BaseModel):
    claim: str

class ClaimCheck(BaseModel):
    claim: str
    verdict: Literal["Supported", "Misleading", "Unsupported"]
    evidence: str = Field(description="Short quote or number from the evidence, with section ID")
    explanation: str = Field(description="One or two sentences a non-expert can understand")

def audit_claims(tree=None, facts=None):
    tree, facts = tree or TREE, facts or FACTS
    ceo_id = next(i for i, n in tree.items() if n["level"] == 2)  # the first section is the CEO letter
    claims = ask_gemini(
        "List every distinct sustainability claim in this text, one short sentence each:\n\n" + node_text(tree, ceo_id),
        types.GenerateContentConfig(response_mime_type="application/json", response_schema=list[Claim]),
    ).parsed

    # The report is short, so we pass every other section as evidence.
    # For a 200-page report, call navigate() for each claim instead.
    seen = {ceo_id}
    evidence = "".join(node_text(tree, i, seen) for i, n in tree.items() if n["level"] == 2)

    prompt = f"""You are a strict but fair greenwashing auditor.
For each claim, decide:
- Supported: the evidence fully backs it.
- Misleading: partly true but leaves out something important, or uses a narrow scope to sound bigger.
- Unsupported: the evidence contradicts it or doesn't back it at all.

COMPUTED FACTS (calculated with code from the report's own data, trust these over your own maths):
{facts}

REPORT EVIDENCE:
{evidence}

CLAIMS:
{chr(10).join("- " + c.claim for c in claims)}"""
    return ask_gemini(prompt, types.GenerateContentConfig(response_mime_type="application/json", response_schema=list[ClaimCheck])).parsed

checks = audit_claims()
ICON = {"Supported": "🟢", "Misleading": "🟠", "Unsupported": "🔴"}
for c in checks:
    print(f"{ICON[c.verdict]} {c.verdict.upper()}: {c.claim}\n   📎 {c.evidence}\n   💡 {c.explanation}\n")

## Step 8: Build the Demo UI with Gradio 🏆
Three tabs: ask the report anything, see the emissions dashboard, and run the greenwashing audit.

In [ ]:
import gradio as gr

def ui_ask(question):
    if not question.strip():
        return "", ""
    try:
        nav, answer = ask_report(question)
    except Exception as e:
        return f"⚠️ Sorry, something went wrong: {e}", ""
    path = "\n".join(f"- **{i}** {TREE[i]['title']}" for i in nav.node_ids)
    return answer, f"**Sections opened:**\n{path}\n\n**Why:** {nav.reasoning}"

def ui_audit():
    try:
        return [[ICON[c.verdict] + " " + c.verdict, c.claim, c.evidence, c.explanation] for c in audit_claims()]
    except Exception as e:
        return [["⚠️ Error", str(e), "", ""]]

with gr.Blocks(title="GreenLens") as demo:
    gr.Markdown("# 🌱 GreenLens\nAn AI analyst for sustainability reports. *(Fictional company: Lumen Logistics)*")
    with gr.Tab("🔎 Ask the report"):
        q = gr.Textbox(label="Question", placeholder="e.g. Is Lumen really carbon neutral?")
        gr.Examples(["Is Lumen really carbon neutral?", "How fast is the fleet going electric?",
                     "Which emissions data has been independently checked?"], inputs=q)
        ask_btn = gr.Button("Ask", variant="primary")
        with gr.Row():
            ans = gr.Markdown(label="Answer")
            trail = gr.Markdown(label="How the AI navigated")
        ask_btn.click(ui_ask, inputs=q, outputs=[ans, trail])
    with gr.Tab("📊 Emissions dashboard"):
        gr.Plot(value=fig)
        gr.Dataframe(value=em.reset_index())
        gr.JSON(value=FACTS, label="Computed facts")
    with gr.Tab("🕵️ Greenwashing audit"):
        audit_btn = gr.Button("Audit CEO claims", variant="primary")
        audit_out = gr.Dataframe(headers=["Verdict", "Claim", "Evidence", "Explanation"], wrap=True)
        audit_btn.click(ui_audit, outputs=audit_out)

# Launch it! (This will generate a public URL you can share with judges)
demo.launch(share=True, theme="soft")

## 🚀 Make It Your Own: Hackathon Extensions
- **Real PDFs:** upload a real company's sustainability report PDF and convert it to markdown with headings, or try the PageIndex SDK for long documents.
- **Compare companies:** run the extraction on 3 competitors and rank them on emissions intensity per employee or per revenue.
- **Consumer version:** scan a product label or ad photo and check its green claims ("eco", "carbon neutral").
- **SME carbon calculator:** upload electricity and fuel bills, extract usage, apply emission factors in pandas, and draft a simple Scope 1 and 2 report for a small business.
- **Supply-chain screener:** a bank checks a loan applicant's report against its own green-loan criteria.

## 🎤 Pitching to Judges
- Show the claim *"cut our footprint by 40%"* next to the chart where the **total goes up**. Instant impact.
- Explain **why no vector DB**: reports are structured documents, and tables must stay whole. This shows depth.
- Show the **validation check** catching the typo: *"we don't just trust the AI, we verify it."*

## 🛡️ Responsible AI Checklist
- [ ] Every verdict shows evidence with a section ID, so a human can check it.
- [ ] Numbers are computed by code, not by the LLM.
- [ ] "Misleading" is an accusation. Treat the output as a **screening tool** for a human analyst, not a published judgement.
- [ ] Note what the report doesn't say (no Scope 3 target, unverified offsets). Absence of evidence matters.